In [5]:
import torch
import torch.nn as nn
import lab07_starter as S

In [6]:
import json

# Load and execute the vae_checks.ipynb file directly
with open('vae_checks.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

for cell in nb['cells']:
    if cell['cell_type'] == 'code':
        code = ''.join(cell['source'])
        exec(code)

print("vae_checks.ipynb loaded successfully! VAE and vae_loss_fn are now available.")

Check 1 - Closed-form KL: 6.5805 | Monte Carlo KL: 6.5815
Check 2 - Encoder grad norm (with trick): 4.6129 | (detached): 0.0000
Check 3 - VAE loss at beta=1.0 runs successfully: 44.5212
vae_checks.ipynb loaded successfully! VAE and vae_loss_fn are now available.


In [8]:
if __name__ == "__main__":
    Xtr, ytr, Xte, yte = S.load_task()
    net = S.train_feature_net(Xtr, ytr)
    
    # Train plain AE and unconditional VAE (beta=1.0) for interpolation comparison
    print("Training Plain AE for interpolation...")
    ae = S.AE(d_latent=8)
    S.train(ae, S.ae_loss, Xtr, epochs=200, seed=0)
    
    print("Training Unconditional VAE for interpolation...")
    vae = VAE(d_latent=8)
    loss_fn_b1 = lambda m, xb, yb: vae_loss_fn(m, xb, yb, beta=1.0)
    S.train(vae, loss_fn_b1, Xtr, epochs=200, seed=0)

    # ==========================================
    # 2. Latent Space Interpolation Study 
    # ==========================================
    pairs = []
    for _ in range(40):
        idx1, idx2 = torch.randint(0, len(Xte), (2,))
        while yte[idx1] == yte[idx2]:
            idx2 = torch.randint(0, len(Xte), (1,))[0]
        pairs.append((Xte[idx1], Xte[idx2]))
        
    @torch.no_grad()
    def evaluate_interpolation(model_type="vae"):
        confidences = []
        class_changes = []
        for x1, x2 in pairs:
            alphas = torch.linspace(0, 1, 10)
            if model_type == "vae":
                mu1, _ = vae.encode(x1.unsqueeze(0))
                mu2, _ = vae.encode(x2.unsqueeze(0))
                path_latent = torch.stack([(1 - a) * mu1 + a * mu2 for a in alphas]).squeeze(1)
                path_imgs = torch.sigmoid(vae.decode(path_latent))
            elif model_type == "ae":
                z1 = ae.encode(x1.unsqueeze(0))
                z2 = ae.encode(x2.unsqueeze(0))
                path_latent = torch.stack([(1 - a) * z1 + a * z2 for a in alphas]).squeeze(1)
                path_imgs = ae.decode(path_latent).clamp(0, 1)
            else: # Pixel cross-fade baseline
                path_imgs = torch.stack([(1 - a) * x1 + a * x2 for a in alphas])
                
            logits = net(path_imgs)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            
            confidences.append(probs.max(dim=1).values.mean().item())
            class_changes.append((preds[1:] != preds[:-1]).sum().item())
        return np.mean(confidences), np.mean(class_changes)

    print("\n=== Piece 4.1: Latent Interpolation Results ===")
    for m in ["vae", "ae", "pixel"]:
        conf, changes = evaluate_interpolation(m)
        print(f"{m.upper()} Path -> Mean Confidence: {conf:.4f} | Mean Class Changes: {changes:.2f}")

    # ==========================================
    # 3. Conditional VAE (CVAE) Evaluation 
    # ==========================================
    class CVAE(nn.Module):
        def __init__(self, d_in=64, d_hidden=64, d_latent=8, n_classes=10):
            super().__init__()
            self.enc = nn.Sequential(nn.Linear(d_in + n_classes, d_hidden), nn.ReLU())
            self.fc_mu = nn.Linear(d_hidden, d_latent)
            self.fc_var = nn.Linear(d_hidden, d_latent)
            
            self.dec = nn.Sequential(
                nn.Linear(d_latent + n_classes, d_hidden),
                nn.ReLU(),
                nn.Linear(d_hidden, d_in)
            )

        def encode(self, x, c):
            h = self.enc(torch.cat([x, c], dim=1))
            return self.fc_mu(h), self.fc_var(h)

        def reparameterize(self, mu, log_var):
            std = torch.exp(0.5 * log_var)
            return mu + torch.randn_like(std) * std

        def decode(self, z, c):
            return self.dec(torch.cat([z, c], dim=1))

        def forward(self, x, c):
            mu, log_var = self.encode(x, c)
            z = self.reparameterize(mu, log_var)
            return self.decode(z, c), mu, log_var

    cvae = CVAE()
    def cvae_loss(model, xb, yb):
        c = S.one_hot(yb)
        logits, mu, log_var = model(xb, c)
        recon = nn.functional.binary_cross_entropy_with_logits(logits, xb, reduction='none').sum(dim=1).mean()
        kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1).mean()
        return recon + kl
        
    print("\nTraining Conditional VAE (CVAE)...")
    S.train(cvae, cvae_loss, Xtr, ytr, epochs=200)
    
    # Generate 200 images per digit from CVAE prior
    with torch.no_grad():
        correct_counts = 0
        total_eval = 0
        for digit in range(10):
            z_sample = torch.randn(200, 8)
            c_sample = S.one_hot(torch.full((200,), digit, dtype=torch.long))
            gen_imgs = torch.sigmoid(cvae.decode(z_sample, c_sample))
            preds = net(gen_imgs).argmax(dim=1)
            correct_counts += (preds == digit).sum().item()
            total_eval += 200
        print(f"\n=== Piece 4.2: CVAE conditional accuracy share: {correct_counts / total_eval:.3f} ===")

Training Plain AE for interpolation...
Training Unconditional VAE for interpolation...

=== Piece 4.1: Latent Interpolation Results ===
VAE Path -> Mean Confidence: 0.9070 | Mean Class Changes: 1.77
AE Path -> Mean Confidence: 0.9152 | Mean Class Changes: 1.77
PIXEL Path -> Mean Confidence: 0.9365 | Mean Class Changes: 1.32

Training Conditional VAE (CVAE)...

=== Piece 4.2: CVAE conditional accuracy share: 1.000 ===
